<a href="https://colab.research.google.com/github/Poojarautela03/ABTALKS/blob/main/day-19-prompt-engineering/prompt-engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 19 · Prompt Engineering for Reliable LLM Outputs

**Focus Area:** Systematic Prompt Engineering

This notebook evaluates five prompt engineering techniques on the same information-extraction task:

1. Role assignment
2. Output format specification
3. Chain-of-thought reasoning
4. Few-shot examples
5. Negative constraints

It also builds a grounding system prompt for a RAG assistant and tests it against five out-of-context questions.

> **API note:** Set `GEMINI_API_KEY` as a Colab secret before running the API cells (used instead of OpenAI due to API quota limits). If no key is available, the notebook automatically uses a deterministic offline evaluation mode so the notebook remains reproducible and uploadable to GitHub.

## 1. Imports and configuration

The experiment uses Python and the Google Gemini API. The default model can be changed with `GEMINI_MODEL`.

The task is **structured information extraction**: extract the document's topic, main entities, sentiment, and a short summary.

In [1]:
import os
import json
import re
import time
import statistics
from typing import Any, Dict, List

import google.generativeai as genai

try:
    from google.colab import userdata
    API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    API_KEY = os.getenv("GEMINI_API_KEY")

MODEL = os.getenv("GEMINI_MODEL", "gemini-flash-lite-latest")

USE_API = bool(API_KEY)

if USE_API:
    genai.configure(api_key=API_KEY)
    chat_model = genai.GenerativeModel(MODEL)
    print(f"Gemini API enabled | model={MODEL}")
else:
    chat_model = None
    print("Gemini API not enabled. Using deterministic offline evaluation mode.")
    print("To use live API evaluation, set GEMINI_API_KEY in Colab secrets and rerun the API cells.")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Gemini API enabled | model=gemini-flash-lite-latest


## 2. Define the task and 10 diverse inputs

### Task
Given a short document, extract:

- `topic`
- `entities` (important named entities)
- `sentiment` (`positive`, `negative`, or `neutral`)
- `summary` (one sentence)

The expected output is deliberately simple so that **accuracy** and **format consistency** can be scored consistently across prompt versions.

In [2]:
inputs = [
    {
        "id": 1,
        "text": "The city library launched a free digital literacy program for senior citizens. Microsoft volunteers will teach basic online safety and document editing every Saturday.",
        "gold": {
            "topic": "digital literacy program",
            "entities": ["city library", "Microsoft"],
            "sentiment": "positive",
            "summary": "The city library launched a free digital literacy program for seniors with Microsoft volunteers teaching online safety and document editing."
        }
    },
    {
        "id": 2,
        "text": "Researchers at the University of Delhi reported that urban rooftop gardens can reduce building surface temperatures during summer. The study recommends native plants and efficient irrigation.",
        "gold": {
            "topic": "urban rooftop gardens",
            "entities": ["University of Delhi"],
            "sentiment": "positive",
            "summary": "University of Delhi researchers found rooftop gardens can reduce building surface temperatures and recommended native plants and efficient irrigation."
        }
    },
    {
        "id": 3,
        "text": "A regional train service was delayed for three hours after heavy rain damaged a section of track near Meerut. Railway officials said repairs were underway.",
        "gold": {
            "topic": "train service disruption",
            "entities": ["Meerut"],
            "sentiment": "negative",
            "summary": "Heavy rain damaged track near Meerut, causing a three-hour train delay while repairs were underway."
        }
    },
    {
        "id": 4,
        "text": "GreenBasket introduced reusable packaging for its grocery deliveries. Customers can return the containers during their next delivery, reducing single-use plastic waste.",
        "gold": {
            "topic": "reusable grocery packaging",
            "entities": ["GreenBasket"],
            "sentiment": "positive",
            "summary": "GreenBasket introduced reusable delivery containers that customers can return to reduce single-use plastic waste."
        }
    },
    {
        "id": 5,
        "text": "The campus robotics club postponed its annual competition because the main laboratory is being renovated. The event will be rescheduled after safety inspections.",
        "gold": {
            "topic": "robotics competition postponement",
            "entities": ["campus robotics club"],
            "sentiment": "negative",
            "summary": "The campus robotics club postponed its annual competition because its main laboratory is undergoing renovation."
        }
    },
    {
        "id": 6,
        "text": "The health department released a report on seasonal influenza cases. Officials noted that case numbers were stable compared with the previous reporting period.",
        "gold": {
            "topic": "seasonal influenza cases",
            "entities": ["health department"],
            "sentiment": "neutral",
            "summary": "The health department reported that seasonal influenza case numbers were stable compared with the previous period."
        }
    },
    {
        "id": 7,
        "text": "SolarSpark opened a community solar project outside Jaipur. Households can subscribe to a share of the electricity generated by the solar panels.",
        "gold": {
            "topic": "community solar project",
            "entities": ["SolarSpark", "Jaipur"],
            "sentiment": "positive",
            "summary": "SolarSpark opened a community solar project outside Jaipur that lets households subscribe to shared solar electricity."
        }
    },
    {
        "id": 8,
        "text": "The museum announced that its ancient coin exhibition will close two weeks earlier than planned because of electrical maintenance work.",
        "gold": {
            "topic": "museum exhibition closure",
            "entities": ["museum"],
            "sentiment": "negative",
            "summary": "The museum's ancient coin exhibition will close two weeks early because of electrical maintenance."
        }
    },
    {
        "id": 9,
        "text": "A startup called AquaSense developed a low-cost sensor that monitors water quality in small reservoirs. The prototype is being tested by local farmers.",
        "gold": {
            "topic": "low-cost water quality sensor",
            "entities": ["AquaSense"],
            "sentiment": "positive",
            "summary": "AquaSense developed a low-cost reservoir water-quality sensor that is being tested by local farmers."
        }
    },
    {
        "id": 10,
        "text": "The university announced that the library will remain open until midnight during examination week. Students will need their identification cards to enter after 9 p.m.",
        "gold": {
            "topic": "extended library hours",
            "entities": ["university"],
            "sentiment": "positive",
            "summary": "The university will keep its library open until midnight during examination week, with ID cards required after 9 p.m."
        }
    }
]

len(inputs)

10

## 3. Prompt versions

Each technique is added **one at a time**. The five versions are therefore directly comparable on the same 10 documents.

> **Important:** For the chain-of-thought condition, the prompt asks the model to reason internally and return only the requested structured result. The notebook does not collect or expose hidden chain-of-thought.

In [3]:
BASELINE = """Extract information from the document.
Return the topic, entities, sentiment, and a one-sentence summary.

Document:
{document}
"""

ROLE = """You are a careful information extraction specialist.
Extract information from the document.
Return the topic, entities, sentiment, and a one-sentence summary.

Document:
{document}
"""

FORMAT = """You are a careful information extraction specialist.
Extract information from the document.

Return ONLY valid JSON with exactly these keys:
{{
  "topic": "short topic",
  "entities": ["entity1", "entity2"],
  "sentiment": "positive | negative | neutral",
  "summary": "one sentence"
}}

Document:
{document}
"""

COT = """You are a careful information extraction specialist.
Analyze the document carefully and reason through the evidence internally before producing the answer.
Do not output your internal reasoning.
Return ONLY valid JSON with exactly these keys:
{{
  "topic": "short topic",
  "entities": ["entity1", "entity2"],
  "sentiment": "positive | negative | neutral",
  "summary": "one sentence"
}}

Document:
{document}
"""

FEWSHOT = """You are a careful information extraction specialist.
Extract information using the same style as the example.

Example document:
"BrightFarm opened a community greenhouse in Noida. Local residents can grow vegetables there."
Example output:
{{"topic":"community greenhouse","entities":["BrightFarm","Noida"],"sentiment":"positive","summary":"BrightFarm opened a community greenhouse in Noida for local residents to grow vegetables."}}

Now extract information from the document below.
Return ONLY valid JSON with exactly these keys:
{{
  "topic": "short topic",
  "entities": ["entity1", "entity2"],
  "sentiment": "positive | negative | neutral",
  "summary": "one sentence"
}}

Document:
{document}
"""

NEGATIVE = """You are a careful information extraction specialist.
Extract information only from the supplied document.

Return ONLY valid JSON with exactly these keys:
{{
  "topic": "short topic",
  "entities": ["entity1", "entity2"],
  "sentiment": "positive | negative | neutral",
  "summary": "one sentence"
}}

Constraints:
- Do not invent facts, entities, causes, dates, or locations.
- Do not use outside knowledge.
- If an entity is not explicitly present, do not add it.
- Keep the summary to exactly one sentence.
- Use only positive, negative, or neutral for sentiment.

Document:
{document}
"""

PROMPTS = {
    "baseline": BASELINE,
    "role_assignment": ROLE,
    "output_format": FORMAT,
    "chain_of_thought": COT,
    "few_shot": FEWSHOT,
    "negative_constraints": NEGATIVE,
}

prompt_descriptions = {
    "baseline": "Simple task instruction with no extra technique.",
    "role_assignment": "Adds an information-extraction specialist role.",
    "output_format": "Adds an explicit JSON schema and fixed fields.",
    "chain_of_thought": "Requests internal reasoning while exposing only the final structured answer.",
    "few_shot": "Adds one input-output example to demonstrate the desired behavior.",
    "negative_constraints": "Adds explicit anti-hallucination and output constraints.",
}

## 4. Model call and robust parsing

If the API is enabled, each prompt version is evaluated by the selected OpenAI model.

For reproducibility without an API key, the notebook includes an offline simulator with deterministic outputs. This makes the GitHub notebook executable without exposing credentials.

In [4]:
def call_model(prompt: str) -> str:
    if not USE_API:
        raise RuntimeError("Live API is disabled.")

    response = chat_model.generate_content(
        prompt,
        generation_config=genai.types.GenerationConfig(temperature=0)
    )
    time.sleep(13)  # stay under Gemini free-tier rate limit
    return response.text.strip()


def parse_json(text: str) -> Dict[str, Any]:
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.S)
        if match:
            try:
                return json.loads(match.group(0))
            except json.JSONDecodeError:
                pass
    return {}


def offline_prediction(item: Dict[str, Any], version: str) -> Dict[str, Any]:
    gold = item["gold"]
    # Simulated errors reflect common prompt-engineering failure modes.
    pred = dict(gold)
    pred["entities"] = list(gold["entities"])

    if version == "baseline":
        if item["id"] in {3, 8}:
            pred["sentiment"] = "neutral"
        if item["id"] in {2, 7}:
            pred["entities"] = []
    elif version == "role_assignment":
        if item["id"] == 8:
            pred["sentiment"] = "neutral"
        if item["id"] == 2:
            pred["entities"] = []
    elif version == "output_format":
        if item["id"] == 8:
            pred["sentiment"] = "neutral"
    elif version == "chain_of_thought":
        if item["id"] == 6:
            pred["sentiment"] = "positive"
    elif version == "few_shot":
        if item["id"] == 6:
            pred["sentiment"] = "positive"
    elif version == "negative_constraints":
        pass

    return pred


def predict(item: Dict[str, Any], version: str) -> Dict[str, Any]:
    prompt = PROMPTS[version].format(document=item["text"])
    if USE_API:
        raw = call_model(prompt)
        parsed = parse_json(raw)
        return {
            "prediction": parsed,
            "raw": raw,
            "prompt": prompt
        }
    else:
        return {
            "prediction": offline_prediction(item, version),
            "raw": json.dumps(offline_prediction(item, version)),
            "prompt": prompt
        }

## 5. Evaluation rubric

Each output receives two scores from **1 to 5**:

### Accuracy
- **5** = all key facts are correct
- **4** = one minor factual/detail issue
- **3** = noticeable but partly correct error
- **2** = several important errors
- **1** = mostly incorrect

### Format consistency
- **5** = exact requested structure and valid fields
- **4** = minor formatting deviation
- **3** = usable but inconsistent structure
- **2** = difficult to parse/use
- **1** = unusable format

For automated evaluation, the scores are derived from field-level matching and schema compliance.

In [5]:
EXPECTED_KEYS = {"topic", "entities", "sentiment", "summary"}

def normalize_text(x: Any) -> str:
    return re.sub(r"\s+", " ", str(x).strip().lower())

def field_match(pred: Any, gold: Any) -> bool:
    if isinstance(gold, list):
        if not isinstance(pred, list):
            return False
        return {normalize_text(x) for x in pred} == {normalize_text(x) for x in gold}
    return normalize_text(pred) == normalize_text(gold)

def accuracy_score(pred: Dict[str, Any], gold: Dict[str, Any]) -> int:
    if not isinstance(pred, dict) or not pred:
        return 1

    matches = sum(field_match(pred.get(k), gold[k]) for k in EXPECTED_KEYS)
    # 4 fields -> map 0..4 matches to 1..5.
    return max(1, min(5, matches + 1))

def format_score(pred: Dict[str, Any]) -> int:
    if not isinstance(pred, dict) or not pred:
        return 1

    score = 5
    if set(pred.keys()) != EXPECTED_KEYS:
        score -= 2
    if not isinstance(pred.get("entities"), list):
        score -= 1
    if pred.get("sentiment") not in {"positive", "negative", "neutral"}:
        score -= 1
    if not isinstance(pred.get("summary"), str):
        score -= 1
    return max(1, score)

def evaluate_version(version: str) -> List[Dict[str, Any]]:
    rows = []
    for item in inputs:
        result = predict(item, version)
        pred = result["prediction"]
        acc = accuracy_score(pred, item["gold"])
        fmt = format_score(pred)
        rows.append({
            "id": item["id"],
            "version": version,
            "accuracy": acc,
            "format_consistency": fmt,
            "average": round((acc + fmt) / 2, 2),
            "prediction": pred
        })
    return rows


all_results = {}
for version in PROMPTS:
    all_results[version] = evaluate_version(version)

print("Evaluation completed for", len(PROMPTS), "prompt versions.")

Evaluation completed for 6 prompt versions.


## 6. Evaluate the baseline on 10 inputs

In [6]:
baseline_results = all_results["baseline"]

for row in baseline_results:
    print(
        f"Input {row['id']:>2}: "
        f"accuracy={row['accuracy']}/5, "
        f"format={row['format_consistency']}/5, "
        f"average={row['average']}/5"
    )

baseline_avg = round(statistics.mean(r["average"] for r in baseline_results), 2)
print(f"\nBaseline average score: {baseline_avg}/5")

Input  1: accuracy=1/5, format=1/5, average=1.0/5
Input  2: accuracy=1/5, format=1/5, average=1.0/5
Input  3: accuracy=1/5, format=1/5, average=1.0/5
Input  4: accuracy=1/5, format=1/5, average=1.0/5
Input  5: accuracy=1/5, format=1/5, average=1.0/5
Input  6: accuracy=1/5, format=1/5, average=1.0/5
Input  7: accuracy=1/5, format=1/5, average=1.0/5
Input  8: accuracy=1/5, format=1/5, average=1.0/5
Input  9: accuracy=1/5, format=1/5, average=1.0/5
Input 10: accuracy=1/5, format=1/5, average=1.0/5

Baseline average score: 1.0/5


## 7. Compare all five techniques

The requested five techniques are evaluated on exactly the same 10-input set.

In [7]:
summary = []

for version, rows in all_results.items():
    avg_accuracy = statistics.mean(r["accuracy"] for r in rows)
    avg_format = statistics.mean(r["format_consistency"] for r in rows)
    avg_score = statistics.mean(r["average"] for r in rows)
    improvement = avg_score - baseline_avg

    summary.append({
        "version": version,
        "description": prompt_descriptions[version],
        "avg_accuracy": round(avg_accuracy, 2),
        "avg_format_consistency": round(avg_format, 2),
        "avg_score": round(avg_score, 2),
        "improvement_vs_baseline": round(improvement, 2)
    })

for row in summary:
    print(
        f"{row['version']:20s} | "
        f"accuracy={row['avg_accuracy']:.2f} | "
        f"format={row['avg_format_consistency']:.2f} | "
        f"overall={row['avg_score']:.2f} | "
        f"Δ baseline={row['improvement_vs_baseline']:+.2f}"
    )

baseline             | accuracy=1.00 | format=1.00 | overall=1.00 | Δ baseline=+0.00
role_assignment      | accuracy=1.00 | format=1.00 | overall=1.00 | Δ baseline=+0.00
output_format        | accuracy=2.30 | format=5.00 | overall=3.65 | Δ baseline=+2.65
chain_of_thought     | accuracy=2.60 | format=5.00 | overall=3.80 | Δ baseline=+2.80
few_shot             | accuracy=3.10 | format=5.00 | overall=4.05 | Δ baseline=+3.05
negative_constraints | accuracy=2.50 | format=4.60 | overall=3.55 | Δ baseline=+2.55


## 8. Prompt version dictionary

This dictionary stores every version, its average evaluation score, and a one-line description of what changed relative to the previous version.

In [8]:
prompt_version_dictionary = {}

previous = None
for row in summary:
    version = row["version"]
    change = prompt_descriptions[version]
    if previous is not None:
        change = f"Added {version.replace('_', ' ')} on top of the previous prompt."

    prompt_version_dictionary[version] = {
        "average_eval_score": row["avg_score"],
        "description": change
    }
    previous = version

print(json.dumps(prompt_version_dictionary, indent=2))

{
  "baseline": {
    "average_eval_score": 1.0,
    "description": "Simple task instruction with no extra technique."
  },
  "role_assignment": {
    "average_eval_score": 1.0,
    "description": "Added role assignment on top of the previous prompt."
  },
  "output_format": {
    "average_eval_score": 3.65,
    "description": "Added output format on top of the previous prompt."
  },
  "chain_of_thought": {
    "average_eval_score": 3.8,
    "description": "Added chain of thought on top of the previous prompt."
  },
  "few_shot": {
    "average_eval_score": 4.05,
    "description": "Added few shot on top of the previous prompt."
  },
  "negative_constraints": {
    "average_eval_score": 3.55,
    "description": "Added negative constraints on top of the previous prompt."
  }
}


## 9. Identify the best technique and form a hypothesis

The best technique is selected by the highest average evaluation score. If multiple techniques tie, the one with higher accuracy is preferred.

In [9]:
best_row = sorted(
    summary,
    key=lambda x: (x["avg_score"], x["avg_accuracy"], x["avg_format_consistency"]),
    reverse=True
)[0]

best_technique = best_row["version"]

hypotheses = {
    "role_assignment": "The specialist role can improve attention to extraction requirements, but it does not by itself force a stable output schema.",
    "output_format": "A strict schema reduces ambiguity about what must be returned, making structured extraction more consistent.",
    "chain_of_thought": "Internal reasoning can help with multi-step interpretation, but the task is simple enough that extra reasoning may add little benefit.",
    "few_shot": "An example demonstrates the desired input-output behavior, reducing ambiguity and helping the model imitate the target structure.",
    "negative_constraints": "Explicit anti-hallucination and formatting constraints reduce unsupported additions and make the model stay closer to the supplied evidence.",
}

print("Best technique:", best_technique)
print("Average score:", best_row["avg_score"], "/ 5")
print("Hypothesis:", hypotheses[best_technique])

Best technique: few_shot
Average score: 4.05 / 5
Hypothesis: An example demonstrates the desired input-output behavior, reducing ambiguity and helping the model imitate the target structure.


## 10. Grounding system prompt for a RAG assistant

The following system prompt explicitly tells the model to answer **only from the supplied context** and refuse when the context does not support an answer.

In [10]:
GROUNDING_SYSTEM_PROMPT = """You are a grounded RAG assistant.

Answer the user's question ONLY using facts explicitly supported by the provided context.

Rules:
1. Do not use outside knowledge, memory, or assumptions.
2. If the context does not contain enough information to answer the question, refuse.
3. When refusing, say exactly: "I cannot answer that from the provided context."
4. Do not guess or fill missing details.
5. Keep answers concise and directly supported by the context.

Context:
{context}
"""

rag_context = """
Project Aurora is a university project that uses a document retrieval system.
The project stores document embeddings in a vector index.
The team evaluated retrieval quality using ten test queries.
"""

## 11. Five out-of-context grounding tests

These questions are deliberately unrelated to the supplied context. A grounded assistant should refuse all five instead of answering from general training knowledge.

When the API is unavailable, the notebook uses a deterministic simulator to demonstrate the expected grounding decision.

In [11]:
out_of_context_questions = [
    "Who is the current President of the United States?",
    "What is the capital of France?",
    "Who invented the telephone?",
    "What is the boiling point of water at sea level?",
    "Which company created the Android operating system?"
]

def grounded_model_answer(question: str) -> str:
    prompt = GROUNDING_SYSTEM_PROMPT.format(context=rag_context)

    if USE_API:
        full_prompt = f"{prompt}\n\nQuestion: {question}"
        response = chat_model.generate_content(
            full_prompt,
            generation_config=genai.types.GenerationConfig(temperature=0)
        )
        time.sleep(13)  # stay under Gemini free-tier rate limit
        return response.text.strip()

    return "I cannot answer that from the provided context."

def refusal_correctly_detected(answer: str) -> bool:
    return normalize_text(answer) == normalize_text(
        "I cannot answer that from the provided context."
    )

grounding_results = []

for i, question in enumerate(out_of_context_questions, start=1):
    answer = grounded_model_answer(question)
    refused = refusal_correctly_detected(answer)
    grounding_results.append({
        "test": i,
        "question": question,
        "answer": answer,
        "refused_correctly": refused
    })

    status = "REFUSED CORRECTLY" if refused else "FAILED — ANSWERED FROM TRAINING/OTHER KNOWLEDGE"
    print(f"Test {i}: {status}")
    print("Question:", question)
    print("Answer:", answer)
    print("-" * 80)

Test 1: REFUSED CORRECTLY
Question: Who is the current President of the United States?
Answer: I cannot answer that from the provided context.
--------------------------------------------------------------------------------
Test 2: REFUSED CORRECTLY
Question: What is the capital of France?
Answer: I cannot answer that from the provided context.
--------------------------------------------------------------------------------
Test 3: REFUSED CORRECTLY
Question: Who invented the telephone?
Answer: I cannot answer that from the provided context.
--------------------------------------------------------------------------------
Test 4: REFUSED CORRECTLY
Question: What is the boiling point of water at sea level?
Answer: I cannot answer that from the provided context.
--------------------------------------------------------------------------------
Test 5: REFUSED CORRECTLY
Question: Which company created the Android operating system?
Answer: I cannot answer that from the provided context.
-----

## 12. Final experiment summary

The following section prints a compact submission-ready summary.

In [12]:
print("PROMPT ENGINEERING EXPERIMENT")
print("=" * 60)

for row in summary:
    print(
        f"{row['version']:20s} "
        f"overall={row['avg_score']:.2f}/5 | "
        f"accuracy={row['avg_accuracy']:.2f}/5 | "
        f"format={row['avg_format_consistency']:.2f}/5"
    )

print("\nBest technique:", best_technique)
print("Best score:", best_row["avg_score"], "/5")
print("Baseline score:", baseline_avg, "/5")
print("Improvement:", round(best_row["avg_score"] - baseline_avg, 2), "points")

grounding_success = sum(r["refused_correctly"] for r in grounding_results)
print(f"Grounding tests passed: {grounding_success}/{len(grounding_results)}")

PROMPT ENGINEERING EXPERIMENT
baseline             overall=1.00/5 | accuracy=1.00/5 | format=1.00/5
role_assignment      overall=1.00/5 | accuracy=1.00/5 | format=1.00/5
output_format        overall=3.65/5 | accuracy=2.30/5 | format=5.00/5
chain_of_thought     overall=3.80/5 | accuracy=2.60/5 | format=5.00/5
few_shot             overall=4.05/5 | accuracy=3.10/5 | format=5.00/5
negative_constraints overall=3.55/5 | accuracy=2.50/5 | format=4.60/5

Best technique: few_shot
Best score: 4.05 /5
Baseline score: 1.0 /5
Improvement: 3.05 points
Grounding tests passed: 5/5


## 13. Export results

The experiment results are saved as JSON so the notebook has a lightweight machine-readable artifact for GitHub.

In [13]:
export_data = {
    "task": "Structured information extraction",
    "model": MODEL if USE_API else "offline_deterministic_evaluation",
    "prompt_version_dictionary": prompt_version_dictionary,
    "summary": summary,
    "best_technique": best_technique,
    "best_technique_hypothesis": hypotheses[best_technique],
    "grounding_system_prompt": GROUNDING_SYSTEM_PROMPT,
    "grounding_results": grounding_results
}

with open("day19_prompt_engineering_results.json", "w", encoding="utf-8") as f:
    json.dump(export_data, f, indent=2, ensure_ascii=False)

print("Saved: day19_prompt_engineering_results.json")

Saved: day19_prompt_engineering_results.json
